# Q-OPT Quickstart

Run this after:
```
(base) $ conda activate qgss
(qgss) $ jupyter notebook
```
and `pip install -r requirements.txt && pip install -e .` from the repo root.

This notebook builds one small QUBO instance, solves it with every classical baseline plus QAOA (p=1,2,3) on Aer, and validates against the real `FakeMarrakesh` noise model -- all without touching real QPU time.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from qopt.network.topology import get_topology
from qopt.network.traffic import sample_requests
from qopt.optimization.qubo import build_options, build_qubo
from qopt.optimization.classical_baselines import solve_exact, solve_greedy, solve_simulated_annealing
from qopt.optimization.milp_baseline import solve_milp
from qopt.optimization.qaoa_qiskit import run_qaoa_aer

G = get_topology("A", seed=0)
reqs = sample_requests(list(G.nodes()), n_requests=2, seed=1)
link_state = {frozenset(e): dict(qber=0.02, risk=0.15, key_pool_frac=0.7) for e in G.edges()}
options = build_options(G, reqs, link_state, k_paths=2)
problem = build_qubo(options, reqs)
print(f"n_qubits = {len(options)}")

In [ ]:
exact = solve_exact(problem)
greedy = solve_greedy(problem)
sa = solve_simulated_annealing(problem, seed=0)
milp = solve_milp(problem)

for r in [exact, greedy, sa, milp]:
    print(f"{r.solver:20s} obj={r.objective:8.4f}  violations={r.constraint_violations}  runtime={r.runtime_s*1e3:7.2f} ms")

In [ ]:
for p in [1, 2, 3]:
    r = run_qaoa_aer(problem, reps=p, shots=1024, maxiter=60, seed=0, optimal_energy=exact.objective)
    print(f"QAOA p={p}: obj={r.objective:.4f}  AR={r.approximation_ratio:.3f}  violations={r.constraint_violations}")

## Validate against the real ibm_marrakesh noise model (still 0 real QPU time)

In [ ]:
from qiskit_ibm_runtime.fake_provider import FakeMarrakesh
from qiskit_aer.noise import NoiseModel

noise_model = NoiseModel.from_backend(FakeMarrakesh())
r_noisy = run_qaoa_aer(problem, reps=1, shots=1024, maxiter=40, seed=0,
                        noise_model=noise_model, optimal_energy=exact.objective)
print(f"QAOA p=1 under FakeMarrakesh noise: obj={r_noisy.objective:.4f}  AR={r_noisy.approximation_ratio:.3f}")

## Optional: real ibm_marrakesh hardware (spends your QPU time)

See `README.md` section 5 for the full step-by-step. Short version, once you have run
`experiments/e_hw_prepare_marrakesh_job.py` from a terminal (Jupyter's own terminal works fine too):

```python
from qiskit_ibm_runtime import QiskitRuntimeService
QiskitRuntimeService.save_account(
    token="my_api_key", instance="my_crn", overwrite=True, set_as_default=True,
)
service = QiskitRuntimeService()
backends = service.backends()
print(f"Account OK. {len(backends)} backend(s) available:")
for b in backends[:5]:
    print(f"  {b.name} ({b.num_qubits} qubits)")
```

then, from a terminal (Jupyter -> New -> Terminal, with `qgss` active):

```bash
python scripts/run_on_ibm_marrakesh.py \
    --problem-json results/qaoa_hw_problem.json \
    --params-json  results/qaoa_hw_optimal_params.json \
    --reps 1 --shots 2000 --dry-run   # check the time estimate first
```